💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ Sequential         │ 56.7 K │ train │     0 │
│ 1 │ val_acc │ MulticlassAccuracy │      0 │ train │     0 │
└───┴─────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 56.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 56.7 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 12                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\siddh\miniconda3\envs\spyder-env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:
434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\siddh\miniconda3\envs\spyder-env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:
434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [16]:
import torch

a = torch.arange(9).view(3,3).repeat(32,5,1,1)
b = torch.arange(9).view(3,3).repeat(32,5,1,1)

a*b

tensor([[[[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]]],


        [[[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]]],


        [[[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          [ 9, 16, 25],
          [36, 49, 64]],

         [[ 0,  1,  4],
          

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchmetrics.classification import MulticlassAccuracy

class DLGN_Conv_1(L.LightningModule):
    def __init__(self, in_channels, hidden_channels, out_channels, kernel_size, beta, 
                 num_layers, criterion, optimizer, lr, weight_decay, padding='same'):
        super().__init__()
        self.save_hyperparameters()
        
        self.beta = beta
        self.criterion = criterion
        self.lr = lr
        self.optim_class = optimizer
        self.weight_decay = weight_decay
        self.num_layers = num_layers

        self.gating_layers = nn.ModuleList()
        self.value_layers = nn.ModuleList()
        self.U1 = nn.Parameter(torch.randn(hidden_channels))

    #    self.value_layers.append(nn.Conv2d(in_channels, hidden_channels, kernel_size=kernel_size, padding='same', bias=False))

        for _ in range(num_layers):
            self.value_layers.append(nn.Conv2d(hidden_channels, hidden_channels, kernel_size = kernel_size, bias = False))
        self.U_l_p_1 = nn.Linear(hidden_channels,out_channels)

        self.gating_layers.append(nn.Conv2d(in_channels,hidden_channels, kernel_size = kernel_size, bias = False))
        for _ in range(num_layers - 1):
            self.gating_layers.append(nn.Conv2d(hidden_channels,hidden_channels,kernel_size=kernel_size, bias = False))
        

       
        
        self.train_acc = MulticlassAccuracy(num_classes= out_channels)
        self.val_acc = MulticlassAccuracy(num_classes= out_channels)

    def forward(self, x):
        

       # I = torch.eye(x.size()[-2:]).repeat(x.size(0),1,1)
        v = self.U1.view(1,-1,1,1)                # M
        g = x                                       # Nx1x28x28 

        g = self.gating_layers[0](g)                # NxMx26x26
        gated_out = torch.sigmoid(self.beta*g)      # NxMx26x26
        h = gated_out*v                             # NxMx26x26
        
        for i in range(1,self.num_layers - 1):
            g = self.gating_layers[i](g)            # NxMx24x24
            gated_out = torch.sigmoid(self.beta*g)  # NxMx24x24
            v = self.value_layers[i](h)             # NxMx24x24
            h = gated_out * v                       # NxMx24x24
        

        # ...
        # ...
        # h: NxMx14

        



            
  
        h_L = F.adaptive_avg_pool2d(h, (1, 1))      # NxMx1x1
        h_L = torch.flatten(h_L, 1)                 # NxM
        out = self.U_l_p_1(h_L)                     # NxOut_Cls

 
        
        
        return out

    def _shared_step(self, batch):
        x, y = batch
        output = self(x)
        loss = self.criterion(output, y)
        preds = torch.argmax(output, dim=1)
        return loss, preds, y

    def training_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.train_acc(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.val_acc(preds, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return self.optim_class(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)




class DLGN_Conv_2(L.LightningModule):
    def __init__(self, in_channels, hidden_channels, out_channels, kernel_size, beta, 
                 num_conv_layers, num_linear_layers, linear_hidden_dim, 
                 criterion, optimizer, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        
        self.beta = beta
        self.criterion = criterion
        self.lr = lr
        self.optim_class = optimizer
        self.weight_decay = weight_decay
        self.num_conv_layers = num_conv_layers
        self.num_linear_layers = num_linear_layers

        # --- Convolutional DLGN Section ---
        self.gating_convs = nn.ModuleList()
        self.value_convs = nn.ModuleList()
        
        self.U1_conv = nn.Parameter(torch.randn(hidden_channels))

        # Gating path starts from input channels
        self.gating_convs.append(nn.Conv2d(in_channels, hidden_channels, kernel_size=kernel_size, padding='same', bias=False))
        # Value path starts after U1, so we need num_conv_layers - 1 convs to match gating steps
        for _ in range(num_conv_layers - 1):
            self.value_convs.append(nn.Conv2d(hidden_channels, hidden_channels, kernel_size=kernel_size, padding='same', bias=False))
            self.gating_convs.append(nn.Conv2d(hidden_channels, hidden_channels, kernel_size=kernel_size, padding='same', bias=False))

        # --- Linear DLGN Section ---
        self.gating_linears = nn.ModuleList()
        self.value_linears = nn.ModuleList()
        
        # After global average pooling, the dimension is 'hidden_channels'
        current_dim = hidden_channels
        
        for i in range(num_linear_layers):
            self.gating_linears.append(nn.Linear(current_dim, linear_hidden_dim, bias=False))
            self.value_linears.append(nn.Linear(current_dim, linear_hidden_dim, bias=False))
            current_dim = linear_hidden_dim

        # Final prediction layer
        self.classifier_head = nn.Linear(current_dim, out_channels, bias=False)
        
        self.train_acc = MulticlassAccuracy(num_classes=out_channels)
        self.val_acc = MulticlassAccuracy(num_classes=out_channels)

    def forward(self, x):
        # 1. Convolutional DLGN Block
        # Initial gating
        g_conv = self.gating_convs[0](x)
        gate_conv = torch.sigmoid(self.beta * g_conv)
        
        # Initial value is the parameter U1 expanded to spatial dims
        # Shape: [Batch, Hidden, H, W]
        h = gate_conv * self.U1_conv.view(1, -1, 1, 1)

        for i in range(self.num_conv_layers - 1):
            g_conv = self.gating_convs[i+1](g_conv)
            gate_conv = torch.sigmoid(self.beta * g_conv)
            
            v_conv = self.value_convs[i](h)
            h = gate_conv * v_conv

        # 2. Transition: Global Pooling to Linear
        h_linear = F.adaptive_avg_pool2d(h, (1, 1))
        h_linear = torch.flatten(h_linear, 1) # Shape: [Batch, hidden_channels]
        
        # We also need a gating signal for the linear layers
        # Usually, we pass the last gating features or the pooled features
        g_linear = h_linear 

        # 3. Linear DLGN Block
        for i in range(self.num_linear_layers):
            # Compute gate from the gating path
            g_gate_out = self.gating_linears[i](g_linear)
            gate_val = torch.sigmoid(self.beta * g_gate_out)
            
            # Compute value from the value path
            v_linear_out = self.value_linears[i](h_linear)
            
            # Element-wise gating
            h_linear = gate_val * v_linear_out
            g_linear = g_gate_out # Update gating signal for next layer depth

        # 4. Final Output
        return self.classifier_head(h_linear)

    def _shared_step(self, batch):
        x, y = batch
        output = self(x)
        loss = self.criterion(output, y)
        preds = torch.argmax(output, dim=1)
        return loss, preds, y

    def training_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.train_acc(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.val_acc(preds, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return self.optim_class(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)


class DLGN_Conv_BN(L.LightningModule):
    def __init__(self, in_channels, hidden_channels, out_channels, kernel_size, beta, 
                 num_layers, linear_hidden_dim, num_linear_layers, img_size,
                 criterion, optimizer, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        
        self.beta = beta
        self.num_layers = num_layers
        self.num_linear_layers = num_linear_layers

        # --- Convolutional DLGN Section ---
        self.gating_convs = nn.ModuleList()
        self.value_convs = nn.ModuleList()
        self.gate_bns = nn.ModuleList()  # BN for gating path
        self.value_bns = nn.ModuleList() # BN for value path

        # Initial Layer
        self.U1 = nn.Parameter(torch.randn(hidden_channels))
        self.gating_convs.append(nn.Conv2d(in_channels, hidden_channels, kernel_size, padding='same', bias=False))
        self.gate_bns.append(nn.BatchNorm2d(hidden_channels))

        self.criterion = criterion
        self.optim_class = optimizer
        self.lr = lr
        self.weight_decay = weight_decay
        self.img_size = img_size

        # Subsequent Convolutional Layers
        for _ in range(num_layers - 1):
            self.gating_convs.append(nn.Conv2d(hidden_channels, hidden_channels, kernel_size, padding='same', bias=False))
            self.gate_bns.append(nn.BatchNorm2d(hidden_channels))
            
            self.value_convs.append(nn.Conv2d(hidden_channels, hidden_channels, kernel_size, padding='same', bias=False))
            self.value_bns.append(nn.BatchNorm2d(hidden_channels))

        # --- Linear DLGN Section ---
        self.gating_linears = nn.ModuleList()
        self.value_linears = nn.ModuleList()
        self.gate_linear_bns = nn.ModuleList()
        self.value_linear_bns = nn.ModuleList()

        # Assuming CIFAR-10 (32x32) with 'same' padding
        current_dim = hidden_channels * img_size * img_size

        for i in range(num_linear_layers):
            self.gating_linears.append(nn.Linear(current_dim, linear_hidden_dim, bias=False))
            self.gate_linear_bns.append(nn.BatchNorm1d(linear_hidden_dim))
            
            self.value_linears.append(nn.Linear(current_dim, linear_hidden_dim, bias=False))
            self.value_linear_bns.append(nn.BatchNorm1d(linear_hidden_dim))
            current_dim = linear_hidden_dim

        self.classifier_head = nn.Linear(current_dim, out_channels, bias=False)
        
        self.train_acc = MulticlassAccuracy(num_classes=out_channels)
        self.val_acc = MulticlassAccuracy(num_classes=out_channels)

    def forward(self, x):
        # 1. First Conv Layer (Gate only, Value is U1)
        g = self.gate_bns[0](self.gating_convs[0](x))
        gate_out = torch.sigmoid(self.beta * g)
        h = gate_out * self.U1.view(1, -1, 1, 1)

        # 2. Remaining Conv Layers
        for i in range(self.num_layers - 1):
            # Update Gate
            g = self.gate_bns[i+1](self.gating_convs[i+1](g))
            gate_out = torch.sigmoid(self.beta * g)
            
            # Update Value
            v = self.value_bns[i](self.value_convs[i](h))
            h = gate_out * v

        # 3. Flatten for Linear Layers
        h_flat = torch.flatten(h, 1)
        g_flat = h_flat # Seed the linear gating path with flattened spatial features

        # 4. Linear DLGN Block
        for i in range(self.num_linear_layers):
            g_linear = self.gate_linear_bns[i](self.gating_linears[i](g_flat))
            gate_val = torch.sigmoid(self.beta * g_linear)
            
            v_linear = self.value_linear_bns[i](self.value_linears[i](h_flat))
            h_flat = gate_val * v_linear
            g_flat = g_linear # Pass gating signal to next depth

        return self.classifier_head(h_flat)

    # Standard Lightning boilerplate...
    def _shared_step(self, batch):
        x, y = batch
        output = self(x)
        loss = self.criterion(output, y)
        preds = torch.argmax(output, dim=1)
        return loss, preds, y

    def training_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.train_acc(preds, y)
        self.log("train_loss", loss, prog_bar=True, on_step = False, on_epoch= True)
        self.log("train_acc", self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.val_acc(preds, y)
        self.log("val_loss", loss, prog_bar=True, on_step = False, on_epoch= True)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return self.optim_class(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from torch.optim import Adam

# 2. Data setup with a Split


# 1. The Model and System logic


# 2. Data setup
# For MNIST (Grayscale - 1 Channel)
transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize((0.5,), (0.5,)) 
])
full_train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
#full_train_dataset = datasets.CIFAR10(root="data", train=True, download=True, transform=transform)
# Split into 55,000 training images and 5,000 validation images

print(len(full_train_dataset))
train_subset, val_subset = random_split(full_train_dataset, [55000, 5000])

train_loader = DataLoader(train_subset, batch_size=32, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=32, num_workers=0)

# 3. Execution for DLGN_Conv_1
# Note: Removed linear_hidden_dim and num_linear_layers as they are not in DLGN_Conv_1's __init__
# 3. Execution for DLGN_Conv_2 (Hybrid Architecture)
# 3. Execution for DLGN_Conv_BN (Batch Normalized Hybrid)
model = DLGN_Conv_BN(
    in_channels=1,                # MNIST grayscale
    hidden_channels=20,           # Spatial feature depth
    out_channels=10,              # 10 digits
    kernel_size=3,
    beta=1.0,                     # Gating temperature
    num_layers=3,                 # Number of convolutional gating stages
    num_linear_layers=2,          # Number of fully connected gating stages
    linear_hidden_dim=64,   
    img_size = 28,                  # Dimension for the linear gates
    criterion=nn.CrossEntropyLoss(),
    optimizer=Adam,
    lr=0.001,
    weight_decay=1e-5
)

# 4. Trainer Setup
trainer = L.Trainer(
    max_epochs=30,                # BN usually helps models converge faster
    accelerator="auto", 
    devices=1,
    log_every_n_steps=10,
    #limit_train_batches=200,      # Using your test limits
    #limit_val_batches=30
)

# 5. Run Training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


60000


┏━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name             ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ gating_convs     │ ModuleList         │  7.4 K │ train │     0 │
│ 1  │ value_convs      │ ModuleList         │  7.2 K │ train │     0 │
│ 2  │ gate_bns         │ ModuleList         │    120 │ train │     0 │
│ 3  │ value_bns        │ ModuleList         │     80 │ train │     0 │
│ 4  │ criterion        │ CrossEntropyLoss   │      0 │ train │     0 │
│ 5  │ gating_linears   │ ModuleList         │  1.0 M │ train │     0 │
│ 6  │ value_linears    │ ModuleList         │  1.0 M │ train │     0 │
│ 7  │ gate_linear_bns  │ ModuleList         │    256 │ train │     0 │
│ 8  │ value_linear_bns │ ModuleList         │    256 │ train │     0 │
│ 9  │ classifier_head  │ Linear             │    640 │ train │     0 │
│ 10 │ train_acc        │ MulticlassAccuracy │      0 │ train │     0 │
│ 11 │ val_acc          │ MulticlassAccuracy │      0 │ train │     0 │
│    │ other params     │ n/a                │     20 │ n/a   │   n/a │
└────┴──────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 2.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.0 M                                                                                                
Total estimated model params size (MB): 8                                                                          
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from torch.optim import Adam

# 2. Data setup with a Split


# 1. The Model and System logic


# 2. Data setup
# For MNIST (Grayscale - 1 Channel)
transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize((0.5,), (0.5,)) 
])
#full_train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_train_dataset = datasets.CIFAR10(root="data", train=True, download=True, transform=transform)
# Split into 55,000 training images and 5,000 validation images

print(len(full_train_dataset))
train_subset, val_subset = random_split(full_train_dataset, [45000, 5000])

train_loader = DataLoader(train_subset, batch_size=32, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=32, num_workers=0)

# 3. Execution for DLGN_Conv_1
# Note: Removed linear_hidden_dim and num_linear_layers as they are not in DLGN_Conv_1's __init__
model = DLGN_Conv_BN(
    in_channels=3,                # MNIST grayscale
    hidden_channels=20,           # Spatial feature depth
    out_channels=10,              # 10 digits
    kernel_size=3,
    beta=1.0,                     # Gating temperature
    num_layers=3,                 # Number of convolutional gating stages
    num_linear_layers=2,          # Number of fully connected gating stages
    linear_hidden_dim=64,   
    img_size = 32,                  # Dimension for the linear gates
    criterion=nn.CrossEntropyLoss(),
    optimizer=Adam,
    lr=0.001,
    weight_decay=1e-5
)

trainer = L.Trainer(
    max_epochs=25, 
    accelerator="auto", 
    devices=1,
    log_every_n_steps=10,
    limit_train_batches= 200,
    limit_val_batches= 30
          # Recommended for monitoring
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Files already downloaded and verified


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


50000


┏━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name             ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ gating_convs     │ ModuleList         │  7.7 K │ train │     0 │
│ 1  │ value_convs      │ ModuleList         │  7.2 K │ train │     0 │
│ 2  │ gate_bns         │ ModuleList         │    120 │ train │     0 │
│ 3  │ value_bns        │ ModuleList         │     80 │ train │     0 │
│ 4  │ criterion        │ CrossEntropyLoss   │      0 │ train │     0 │
│ 5  │ gating_linears   │ ModuleList         │  1.3 M │ train │     0 │
│ 6  │ value_linears    │ ModuleList         │  1.3 M │ train │     0 │
│ 7  │ gate_linear_bns  │ ModuleList         │    256 │ train │     0 │
│ 8  │ value_linear_bns │ ModuleList         │    256 │ train │     0 │
│ 9  │ classifier_head  │ Linear             │    640 │ train │     0 │
│ 10 │ train_acc        │ MulticlassAccuracy │      0 │ train │     0 │
│ 11 │ val_acc          │ MulticlassAccuracy │      0 │ train │     0 │
│    │ other params     │ n/a                │     20 │ n/a   │   n/a │
└────┴──────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 2.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.6 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [4]:

import torch
from torch.utils.data import DataLoader,TensorDataset,random_split
import lightning as L
import random
import matplotlib.pyplot as plt
from torchvision import transforms


class ImageDataModule(L.LightningDataModule):
    def __init__(self, block_size, grid_size, num_samples, bg_mu, data_mu, tree_depth, train_size, batch_size, control_group = False):
        super().__init__()
        self.save_hyperparameters()
        self.block_size = block_size
        self.grid_size = grid_size
        self.num_samples = num_samples
        self.bg_mu = bg_mu
        self.data_mu = data_mu
        self.depth = tree_depth
        self.train_size = train_size
        self.batch_size = batch_size
        self.control = control_group




    def setup(self, stage=None):
        # 1. Generate the raw data and labels
        self.grids, self.data, self.block_index = self._generate_data(
            self.block_size, self.grid_size, self.num_samples, self.bg_mu, self.data_mu
        )
        # The labels are generated here (likely as Int by default)
        _, self.labels = self._generate_labels(self.data, self.depth)
        
        # 2. FIX: Cast labels to long() right here
        # .long() is an alias for torch.int64
        self.full_dataset = TensorDataset(self.grids, self.labels.long())

        # 3. Handle the split with absolute integers to avoid float errors
        train_len = int(self.num_samples * self.train_size)
        val_len = self.num_samples - train_len
        
        self.train, self.val = random_split(self.full_dataset, [train_len, val_len])
        
    def train_dataloader(self):
        return DataLoader(self.train, batch_size = self.batch_size)
    def val_dataloader(self):
        return DataLoader(self.val,batch_size = self.batch_size )
            


    def _generate_data(self, block_size, grid_size, num_samples, background_mu=0, data_mu=5):
        # 1. Create background
        grid = torch.normal(torch.full((num_samples, grid_size, grid_size), background_mu, dtype=torch.float32))
        
        # 2. Correct block indexing
        blocks_per_side = grid_size // block_size
        num_blocks = blocks_per_side**2
        block_index = torch.randint(high=num_blocks, size=(num_samples,))
        
        # Calculate top-left corner of the block
        # block_x is the column (0, 3, 6, 9...)
        # block_y is the row (0, 3, 6, 9...)
        block_xs = (block_index % blocks_per_side) * block_size
        block_ys = (block_index // blocks_per_side) * block_size
        
        # 3. Create relative meshgrid for block offsets
        add_x, add_y = torch.meshgrid(torch.arange(block_size), torch.arange(block_size), indexing='ij')
        
        # 4. Expand dimensions for broadcasting
        id_x = block_xs.view(-1, 1, 1) + add_x
        id_y = block_ys.view(-1, 1, 1) + add_y
        batch_id = torch.arange(num_samples).view(-1, 1, 1).expand(-1, block_size, block_size)

        # 5. Generate and place data
        data = torch.normal(torch.full((num_samples, block_size, block_size), data_mu, dtype=torch.float32))
        grid[batch_id, id_y, id_x] = data

        return grid.unsqueeze(1), data, block_index
    def _generate_labels(self,data,depth = 4):

        data_size = data.size(1)*data.size(2)
        num_samples = data.size(0)

        # initialize weights for each dimension

        weights = torch.eye(data_size,2**depth - 1,dtype = torch.float32)

        # initialize tree indices for each sample

        index = torch.ones(num_samples, dtype = torch.int)

        # flatten and mean center data
        data -= torch.mean(data,dim = (1,2), keepdim= True,dtype = torch.float32)
        data = torch.flatten(data,1,-1)



        # traverse till leaf
        while torch.all(index <= 2**(depth - 1) - 1):
            condition =  (data @ weights[:,index - 1] >= 0).diag()
            index = 2*(index)*(~condition) + (2*index + 1)*condition

        

        # assign labels
        labels = index % 2

        return weights, labels


In [22]:
import lightning as L
import torch.nn as nn
from torch.optim import Adam

# 1. Initialize the Data Module
# Parameters: block_size, grid_size, num_samples, bg_mu, data_mu, tree_depth, train_size, batch_size
data_module = ImageDataModule(
    block_size=3,      # Size of the "data" square inside the grid
    grid_size=12,      # Total grid size (12x12)
    num_samples=10000, # Total generated images
    bg_mu=0.0,         # Background mean
    data_mu=2.0,       # Data block mean (signal strength)
    tree_depth=4,      # Depth of the labeling decision tree
    train_size=0.8,    # 80% for training, 20% for validation
    batch_size=32
)

# 2. Initialize the Model
# out_channels=2 because labels are binary (index % 2)
model = DLGN_Conv_BN(
    in_channels=1,                # MNIST grayscale
    hidden_channels=20,           # Spatial feature depth
    out_channels=10,              # 10 digits
    kernel_size=6,
    beta=1.0,                     # Gating temperature
    num_layers=3,            # Number of convolutional gating stages
    num_linear_layers=2,          # Number of fully connected gating stages
    linear_hidden_dim=64, 
    img_size = 12,
                    # Dimension for the linear gates
    criterion=nn.CrossEntropyLoss(),
    optimizer=Adam,
    lr=0.001,
    weight_decay=1e-5
)

# 3. Setup the Trainer
trainer = L.Trainer(
    max_epochs=7,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10000,
    limit_train_batches= 200,
    limit_val_batches= 20 # Keeps the graph clean as discussed
)

# 4. Run Training
trainer.fit(model, datamodule=data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name             ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ gating_convs     │ ModuleList         │ 29.5 K │ train │     0 │
│ 1  │ value_convs      │ ModuleList         │ 28.8 K │ train │     0 │
│ 2  │ gate_bns         │ ModuleList         │    120 │ train │     0 │
│ 3  │ value_bns        │ ModuleList         │     80 │ train │     0 │
│ 4  │ criterion        │ CrossEntropyLoss   │      0 │ train │     0 │
│ 5  │ gating_linears   │ ModuleList         │  188 K │ train │     0 │
│ 6  │ value_linears    │ ModuleList         │  188 K │ train │     0 │
│ 7  │ gate_linear_bns  │ ModuleList         │    256 │ train │     0 │
│ 8  │ value_linear_bns │ ModuleList         │    256 │ train │     0 │
│ 9  │ classifier_head  │ Linear             │    640 │ train │     0 │
│ 10 │ train_acc        │ MulticlassAccuracy │      0 │ train │     0 │
│ 11 │ val_acc          │ MulticlassAccuracy │      0 │ train │     0 │
│    │ other params     │ n/a                │     20 │ n/a   │   n/a │
└────┴──────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 436 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 436 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=7` reached.


In [9]:
import matplotlib.pyplot as plt
for w in model.value_layers[0].weight.detach().numpy():
    for i in w:
        plt.imshow(i)
        plt.colorbar()
        plt.show()
    

AttributeError: 'DLGN_Conv_2' object has no attribute 'value_layers'